In [2]:
# ! mamba remove -n base pyimagej
# ! pip install pyimagej

from IPython.display import Image, display
from pathlib import Path
import jpype
import imagej
import scyjava
from scyjava import jimport
import pandas as pd
import numpy as np
import os

print("Dependencies present")

Dependencies present


In [3]:
# Set Random Image for Testing
test_image_pth = "content/Images1/noise 3 21001.0.png"

# Pre-sets
path_in1 = "./content/Images1/"
path_in2 = "./content/Images2/"
path_in3 = "./content/Images3/"

path_out_csv = "./data/feats_extracted.csv"

# Set Java heap size = 6 gb.
# scyjava.config.add_option('-Xmx6g')

# Folder containing PNG images
# Select one of the input paths (path_in1, path_in2, path_in3)
input_dir = path_in1
output_csv = Path(path_out_csv)

print("I/O Set.")

I/O Set.


In [4]:
import platform, subprocess, os, sys
print(platform.platform())
subprocess.run(["/usr/bin/env","bash","-lc","java -version"])

macOS-15.6.1-arm64-arm-64bit


openjdk version "11.0.27" 2025-04-15 LTS
OpenJDK Runtime Environment Zulu11.80+21-CA (build 11.0.27+6-LTS)
OpenJDK 64-Bit Server VM Zulu11.80+21-CA (build 11.0.27+6-LTS, mixed mode)


CompletedProcess(args=['/usr/bin/env', 'bash', '-lc', 'java -version'], returncode=0)

In [5]:
import imagej.doctor
imagej.doctor.checkup()


Checking Python:
--> Python executable = /Users/mikhailblinov/miniconda3/envs/pyij/bin/python

Checking environment:
--> CONDA_PREFIX = /Users/mikhailblinov/miniconda3/envs/pyij
--> Python executable matches Conda environment.

Checking Python dependencies:
--> jgo: /Users/mikhailblinov/miniconda3/envs/pyij/lib/python3.10/site-packages/jgo/__init__.py
--> scyjava: /Users/mikhailblinov/miniconda3/envs/pyij/lib/python3.10/site-packages/scyjava/__init__.py
--> imglyb: /Users/mikhailblinov/miniconda3/envs/pyij/lib/python3.10/site-packages/imglyb/__init__.py
--> pyimagej: /Users/mikhailblinov/miniconda3/envs/pyij/lib/python3.10/site-packages/imagej/__init__.py

Checking Maven:
--> Maven executable = NOT FOUND!
Checking Java:
--> JAVA_HOME = /Library/Java/JavaVirtualMachines/amazon-corretto-8.jdk/Contents/Home
--> Java executable = /Library/Java/JavaVirtualMachines/amazon-corretto-8.jdk/Contents/Home/bin/java
$ java -version
openjdk version "1.8.0_462"
OpenJDK Runtime Environment Corretto-8

In [6]:
!python -m pip freeze | egrep 'ipykernel|traitlets|tornado|jupyter-client|pyzmq'
!python -m pip install --upgrade "tornado==6.1" "jupyter-client==7.3.2"
!python -m pip install --upgrade "ipykernel>=6.29" "traitlets>=5.9" "pyzmq>=25"


ipykernel==7.1.0
pyzmq==27.1.0
tornado==6.5.2
traitlets==5.14.3
  Using cached tornado-6.1-cp310-cp310-macosx_11_0_arm64.whl
  Using cached jupyter_client-7.3.2-py3-none-any.whl.metadata (8.5 kB)
Using cached jupyter_client-7.3.2-py3-none-any.whl (131 kB)
  Attempting uninstall: tornado
    Found existing installation: tornado 6.5.2
    Uninstalling tornado-6.5.2:
      Successfully uninstalled tornado-6.5.2
  Attempting uninstall: jupyter-client
    Found existing installation: jupyter_client 8.6.3
    Uninstalling jupyter_client-8.6.3:
      Successfully uninstalled jupyter_client-8.6.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jupyter-client]m [jupyter-client]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipykernel 7.1.0 requires jupyter-client>=8.0.0, but you have jupyter-client 7.3.2 which is incompatible.
ipykernel 7.1.0 requires tornado>=6.2, bu

In [ ]:
print(f"Analyzing measurements on all images from folder {input_dir}")

# Initialize ImageJ
# Java Virtual Machine Guarding
#if 'ij' not in globals():
ij = imagej.init('/Applications/Fiji')
#ij = imagej.init('sc.fiji:fiji', headless=False, add_legacy=True)
#else:
print("JVM instance already running.")

print(f"Ensure ImageJ Legacy layer is available. Status: {ij.legacy.isActive()}")

counter = 0


for file in os.listdir(input_dir):
    counter += 1
    
    if counter % 1000 == 0:
        print(counter)

    # Can be deleted for production purposes later. Implemented for efficiency.
    if counter > 2000:
        break

    full_path = str(input_dir) + '/' + file

    # Load test image
    dataset = ij.IJ.openImage(full_path)

    # Display the image within Jupyter Lab for intuitive purposes 
    #display(Image(full_path))

    # Select entire image
    ij.IJ.run(dataset, "Select All", "")

    # Set Measurements
    ij.IJ.run("Set Measurements...", """area mean standard modal min centroid center perimeter bounding fit shape feret's integrated median skewness kurtosis area_fraction stack display add redirect=None decimal=3""")

    # Extract Measurements from the image
    ij.IJ.run(dataset, "Measure", "")   

print("Measurments complete.") 
print("Do not close ImageJ Results window before compiling the rest of the program.")
print("Current limitation: do not run this block with all three folders because dynamic folder column addition is not yet implemented. Run it with one folder, save to dataframe, then rerun.")

Analyzing measurements on all images from folder ./content/Images1/
JVM instance already running.


NameError: name 'ij' is not defined